# Rust to Scala

Используем Rust код через библиотеку JNA, которая позволяет запускать код Rust из Java кода.

In [ ]:
// Компилируем Rust при запуске (кэшируем при отсутствии изменений)
import sys.process._
import java.net.URLClassLoader
import java.nio.file.{Files, Paths}

trait RustLib extends Library {
  def add_numbers(a: Int, b: Int): Int
}

// Инициализация
@transient var rustLoader: URLClassLoader = _

var is_loaded = false
def reloadRustLib(): RustLib = {
  if (is_loaded) {
    is_loaded = false
    try { rustLoader.close() } catch { case _: Exception => }
    System.gc() // Помогает JVM освободить ресурсы (но не гарантирует немедленной выгрузки)
  }
  
  val workDir = os.pwd
  val rustName = "rust_hopfield"
  val rustDir = s"$workDir/../src/$rustName"
  val buildDir = s"$workDir/../src/target/release/"

  val oss = System.getProperty("os.name").toLowerCase
  val libFormat = oss match {
    case _ if oss.contains("win") => ".dll"
    case _ if oss.contains("mac") => ".dylib"
    case _ => f".so"
  }
  val libName = s"$rustName$libFormat"
  val libPath = os.Path(s"$buildDir$libName")

  val tmpDir = workDir / "tmp"
  val uniqueLibCopy = tmpDir / s"rust_${System.currentTimeMillis()}$libFormat"
  
  os.copy(libPath, uniqueLibCopy) // Копируем с уникальным именем (обход блокировки файла в Windows)
  
  val urls = Array(new java.net.URL(s"file://${tmpDir.toNIO.toUri}"))
  rustLoader = new URLClassLoader(urls, null)
  
  val rustLib = Native.load(uniqueLibCopy.toString, classOf[RustLib], Map("classloader" -> rustLoader))
  
  println(s"✅ RustLib перезагружена! (Версия: ${System.currentTimeMillis()})")
  is_loaded = true
  return rustLib
}


@transient var rustLib: RustLib = reloadRustLib()

In [ ]:
// Пример вызова Rust-функции
val result = rustLib.add_numbers(1337, 42)
println(s"1337 + 42 = $result (вычислено в Rust!)")
// Вывод: 1337 + 42 = 1379 (вычислено в Rust!)

Native.unregister()